# Part 2: Tool Usage

In [ ]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

SUFFIX = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"

outputs = json.loads(run_cmd([
    "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

---
## Section 2: Tool Usage (~10 min)

Agents become powerful when you give them **tools**. Foundry supports:

| Tool Type | Description | Use Case |
|-----------|-------------|----------|
| **Function Calling** | Call your custom functions | Database queries, calculations, APIs |
| **Web Search** | Search the web in real-time | Market research, current events |
| **File Search** | Search uploaded documents (RAG) | Internal docs, PDFs, specs |
| **Code Interpreter** | Run Python code | Data analysis, visualizations |

In [ ]:
# --- 2.1 Function Calling ---
# Define a Python function, register it as a tool schema, and handle calls via the Responses API.

# The function we want the agent to call
def get_product_price(product_name: str) -> str:
    """Look up the list price for a given product."""
    prices = {
        "XPR Analytical Balance": "$18,450",
        "T50 Titrator": "$12,000",
        "SevenExcellence pH Meter": "$3,500",
        "DSC 3+ Thermal Analyzer": "$45,000",
    }
    return prices.get(product_name, f"Product '{product_name}' not found in catalog.")

# Define the tool schema for the agent
# Note: strict=True requires "additionalProperties": false in the schema
price_tool = FunctionTool(
    name="get_product_price",
    description="Look up the list price for a given product by name.",
    parameters={
        "type": "object",
        "properties": {
            "product_name": {
                "type": "string",
                "description": "The product name to look up, e.g. 'XPR Analytical Balance'",
            }
        },
        "required": ["product_name"],
        "additionalProperties": False,
    },
    strict=True,
)

pricing_agent = project_client.agents.create_version(
    agent_name="pricing-assistant",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a pricing assistant. Use the get_product_price tool "
            "to look up product prices. Always provide the price in your response."
        ),
        tools=[price_tool],
    ),
)

# Send a question — the agent may request a tool call
response = openai_client.responses.create(
    input="How much does the XPR Analytical Balance cost?",
    extra_body={"agent_reference": {"name": pricing_agent.name, "type": "agent_reference"}},
)

# Check if the agent made a function call
tool_calls = [item for item in response.output if item.type == "function_call"]
if tool_calls:
    # Execute the function locally and send the result back
    call = tool_calls[0]
    args = json.loads(call.arguments)
    print(f"🔧 Agent called: {call.name}({args})")
    result = get_product_price(**args)
    print(f"   Result: {result}")

    # Send the tool result back to get the final answer
    response = openai_client.responses.create(
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": result,
        }],
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": pricing_agent.name, "type": "agent_reference"}},
    )

print(f"\n🧑 How much does the XPR Analytical Balance cost?")
print(f"🤖 {response.output_text}")
print(f"\n✅ Function tool agent: {pricing_agent.name} v{pricing_agent.version}")


In [ ]:
# --- 2.2 Web Search Tool ---
# The built-in WebSearchPreviewTool lets agents search the web — no extra resources needed.

web_agent = project_client.agents.create_version(
    agent_name="web-researcher",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a research assistant. Search the web to find current, "
            "factual information. Always cite your sources with URLs."
        ),
        tools=[WebSearchPreviewTool()],
    ),
)

response = openai_client.responses.create(
    input="What are the latest trends in industrial IoT for manufacturing in 2026?",
    extra_body={"agent_reference": {"name": web_agent.name, "type": "agent_reference"}},
)
print("🧑 What are the latest trends in industrial IoT for manufacturing in 2026?\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ Web search agent: {web_agent.name} v{web_agent.version}")


In [ ]:
# --- 2.3 File Search Tool ---
# Upload a document to a vector store, then let the agent search it.

# Create vector store and upload the product spec
vector_store = openai_client.vector_stores.create(name="WorkshopDocs")
spec_path = Path.cwd() / "sample_data" / "product_spec.md"

with spec_path.open("rb") as f:
    openai_client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id, file=f
    )
print(f"📄 Uploaded {spec_path.name} to vector store '{vector_store.name}'")

# Create agent with file search
doc_agent = project_client.agents.create_version(
    agent_name="document-expert",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a product specialist. Use file search to answer questions "
            "about product specifications. Cite specific data from the documents."
        ),
        tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    ),
)

response = openai_client.responses.create(
    input="What is the readability and stabilization time of the XPR balance?",
    extra_body={"agent_reference": {"name": doc_agent.name, "type": "agent_reference"}},
)
print(f"\n🧑 What is the readability and stabilization time of the XPR balance?\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ File search agent: {doc_agent.name} v{doc_agent.version}")
print(f"   Vector store ID: {vector_store.id} (keep this — you'll reuse it in Part 2)")


In [ ]:
# --- 2.4 Code Interpreter ---
# The built-in Code Interpreter lets agents write and run Python code in a sandbox.
# Perfect for data analysis — no custom function tool needed.

from azure.ai.projects.models import CodeInterpreterTool, AutoCodeInterpreterToolParam

# Upload the CSV file for the code interpreter to process
sales_path = Path.cwd() / "sample_data" / "sales_data.csv"
sales_file = openai_client.files.create(purpose="assistants", file=open(sales_path, "rb"))
print(f"📊 Uploaded {sales_path.name} (file ID: {sales_file.id})")

code_agent = project_client.agents.create_version(
    agent_name="data-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a data analyst. Use Code Interpreter to analyze the uploaded CSV file. "
            "Write Python code to answer questions. Always show the code you ran and the results. "
            "Include specific numbers in your answers."
        ),
        tools=[CodeInterpreterTool(container=AutoCodeInterpreterToolParam(file_ids=[sales_file.id]))],
    ),
)

question = "Which product has the highest total revenue, and in which region? Show the top 3 products."
response = openai_client.responses.create(
    input=question,
    extra_body={"agent_reference": {"name": code_agent.name, "type": "agent_reference"}},
)

# Extract and display any code the agent ran
for item in response.output:
    if item.type == "code_interpreter_call":
        print(f"🔧 Code Interpreter ran:\n{item.code}\n")

print(f"🧑 {question}\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ Code Interpreter agent: {code_agent.name} v{code_agent.version}")


---
Proceed to `03-prompts-eval.ipynb`.